In [109]:
import pandas
data = pandas.read_csv("churn.csv")

data = data.drop(columns = [ 'customerID' ])
data = data[data["TotalCharges"] != ' ']
data["TotalCharges"] = data["TotalCharges"].astype(float)

data

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.50,No
7039,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.90,No
7040,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,No
7041,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.60,Yes


In [110]:
data.dtypes

gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges        float64
Churn                object
dtype: object

In [111]:
import sklearn.model_selection

X = data.drop(columns = [ "Churn" ])
y = (data["Churn"] == "Yes").to_numpy()

X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(
    X, y, test_size = 0.1, stratify = y, random_state = 1
)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((6328, 19), (704, 19), (6328,), (704,))

In [112]:
numeric = [ "SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges" ]
categorical = list(set(X_train.columns) - set(numeric))

len(numeric), len(categorical), len(X_train.columns)

(4, 15, 19)

In [113]:
import sklearn.compose
import sklearn.preprocessing

ct = sklearn.compose.ColumnTransformer(
    transformers=[
        ("numeric", 'passthrough', numeric),
        ("categorical", sklearn.preprocessing.OneHotEncoder(drop = 'first'), categorical)
    ]
)
X_train = ct.fit_transform(X_train)
X_test = ct.transform(X_test)

X_train.shape, X_test.shape

((6328, 30), (704, 30))

In [114]:
import sklearn.preprocessing

ss = sklearn.preprocessing.StandardScaler()
X_train_scaled = ss.fit_transform(X_train)
X_test_scaled = ss.transform(X_test)

In [117]:
import tqdm
from utils import estimate_quality

import xgboost
import catboost
import sklearn.svm
import sklearn.tree
import sklearn.ensemble
import sklearn.neighbors
import sklearn.naive_bayes
import sklearn.linear_model
import sklearn.neural_network

models = [
    # neighbors
    sklearn.neighbors.KNeighborsClassifier(n_jobs = -1, n_neighbors = 50),
    sklearn.neighbors.RadiusNeighborsClassifier(n_jobs = -1, radius = 7),

    # naive_bayes
    sklearn.naive_bayes.BernoulliNB(),
    sklearn.naive_bayes.GaussianNB(),

    # linear_model
    sklearn.linear_model.LogisticRegression(),
    
    # svm
    sklearn.svm.SVC(probability = True),
    sklearn.svm.NuSVC(probability = True),
    
    # tree
    sklearn.tree.DecisionTreeClassifier(),
    sklearn.tree.ExtraTreeClassifier(),

    # ensemble
    sklearn.ensemble.BaggingClassifier(n_estimators = 100, random_state = 42, n_jobs = -1),
    sklearn.ensemble.AdaBoostClassifier(n_estimators = 500, algorithm = 'SAMME', random_state = 42),
    sklearn.ensemble.ExtraTreesClassifier(n_estimators = 500, random_state = 42, n_jobs = -1),
    sklearn.ensemble.RandomForestClassifier(n_estimators = 1000, random_state = 42, n_jobs = -1),
    sklearn.ensemble.GradientBoostingClassifier(n_estimators = 1000, random_state = 42),
    sklearn.ensemble.HistGradientBoostingClassifier(random_state = 42),

    # neural_network
    sklearn.neural_network.MLPClassifier(max_iter = 2000),
    
    # xgboost
    xgboost.XGBClassifier(n_jobs = -1, n_estimators = 50, max_depth = 4, device = 'gpu'),

    # catboost
    catboost.CatBoostClassifier(
        iterations = 600,
        depth = 4,
        random_seed = 42,
        loss_function = 'MultiClass',
        devices = '0-3',
        task_type = 'GPU',
        verbose = False
    ),
]

results = []
for model in tqdm.tqdm(models):
    model.fit(X_train_scaled, y_train)
    metrics = estimate_quality(model.predict_proba(X_test_scaled), y_test)
    results.append({ 'model': str(model), **metrics })
pandas.DataFrame(results)

100%|██████████| 18/18 [00:41<00:00,  2.32s/it]


,model,Accuracy,Precision,Recall,AUC-ROC,F1-score,True Positive,True Negative,False Positive,False Negative,True Negative Rate (Specificity),Negative Predictive Value,False Positive Rate,False Discovery Rate
0,"KNeighborsClassifier(n_jobs=-1, n_neighbors=50)",0.802557,0.641176,0.582888,0.800805,0.610644,109,456,61,78,0.882012,0.853933,0.117988,0.358824
1,"RadiusNeighborsClassifier(n_jobs=-1, radius=7)",0.734375,0.000000,0.000000,0.802361,0.000000,0,517,0,187,1.000000,0.734375,0.000000,0.000000
2,BernoulliNB(),0.707386,0.468013,0.743316,0.802387,0.574380,139,359,158,48,0.694391,0.882064,0.305609,0.531987
3,GaussianNB(),0.640625,0.412234,0.828877,0.809659,0.550622,155,296,221,32,0.572534,0.902439,0.427466,0.587766
4,LogisticRegression(),0.799716,0.647436,0.540107,0.822836,0.588921,101,462,55,86,0.893617,0.843066,0.106383,0.352564
5,SVC(probability=True),0.801136,0.671533,0.491979,0.769474,0.567901,92,472,45,95,0.912959,0.832451,0.087041,0.328467
6,NuSVC(probability=True),0.802557,0.690476,0.465241,0.784648,0.555911,87,478,39,100,0.924565,0.826990,0.075435,0.309524
7,DecisionTreeClassifier(),0.721591,0.478261,0.529412,0.659347,0.502538,99,409,108,88,0.791103,0.822938,0.208897,0.521739
8,ExtraTreeClassifier(),0.708807,0.456311,0.502674,0.642539,0.478372,94,405,112,93,0.783366,0.813253,0.216634,0.543689
9,"BaggingClassifier(n_estimators=100, n_jobs=-1,...",0.776989,0.594937,0.502674,0.791371,0.544928,94,453,64,93,0.876209,0.829670,0.123791,0.405063
